# 9.1 ISC-CI Model

The ISC-CI model introducing a mechanism for context inference, based on the key assumption that temporal co-occurrence provides a useful basis for inferring shared context. Specifically, it assumes that (1) objects occurring together in a given context tend to share the properties elicited by that context; (2) these co-occurrence statistics are learned over the course of development; and (3) this implicit knowledge provides a basis for inferring, from a few examples of objects encountered in a new context, both which features are relevant in that context and what other objects are likely to occur in that context.

To make these ideas clear, consider the contexts in which you might encounter different kinds of birds: a bird-watching field trip in science class, a visit to the bird section of the zoo, and a picture book about birds. Each situation involves multiple types of birds (e.g., robins, crows, and ravens) and exposure to multiple bird-related properties (e.g., can-fly, eats-worms, is-bird) in various combinations. After these experiences, encountering a new context in which birds are relevant (e.g., learning that crows and ravens have hollow bones in the bird section of the Natural History museum) is likely to be interpreted as relating specifically to birds and their
properties, implying that other birds like robins may also occur in this new context, and that
they will share similar properties (e.g., robins also have hollow bones).

Conversely, contexts such as a science lesson on aerodynamics, a visit to a flight exhibit at a science museum, and
a film on the history of flight are likely to involve multiple types of flying things (e.g., crows,
airplanes, and butterflies) and flight-related properties (e.g., can-fly, has-wings, seen-in-the-sky). This suggests that a new context involving flying objects such as crows and airplanes (e.g., learning that crows and airplanes are associated with Bernoulli’s principle) likely relates to all things that can fly, implying that other flying things like butterflies may also occur in this new context and, again, share similar properties (e.g., butterflies are also associated with Bernoulli’s principle).

Thus, the properties shared by items encountered in a situation can provide a clue about what the current context is, what properties are currently important, and what other items are likely or unlikely also to be observed. The central hypothesis embodied by the ISC-CI model is that learning such environmental structure can support future inferences about which features might be relevant in novel contexts, based on the distribution of items that co-occur in those contexts. That is, observing that a new context involves a certain set of objects (e.g., both robins and airplanes) provides evidence that certain features will be context-relevant (e.g., can-fly and has-wings), but not
others (e.g., lays-eggs), based on past experience.

Importantly, this process is graded and probabilistic rather than absolute, as any given set of objects can co-occur in different contexts at different frequencies. In particular, features that are broadly true of many objects are less likely to be relevant in a new context than features that are true of the more limited set of objects seen in that context (Griffiths et al., 2010; Xu & Tenenbaum, 2007). This is because there is a low likelihood of observing any particular set of objects in a broad context: there are many animals, but few Corvidae, so it is more likely that a context involving both crows and ravens relates to Corvidae specifically than it is that this context relates to animals in general. This is because the probability of observing both crows and ravens in the context of animals is lower than the probability of oberseving both crows and ravens in the context of Corvidae.

*Setup and Installation:*

In [2]:
%%capture
%pip install psyneulink

import psyneulink as pnl
import pandas as pd

## Generating the Training Data

### Feature Co-occurrences

We design a training environment that simulates experiencing object co-occurrences throughout learning under the key assumption noted just above. The environment consisted of a series of episodes corresponding to different contexts. Each context involved a set of objects that share a common semantic feature (e.g., things that are birds, things that can fly, things that are found in the zoo, etc.), with each feature represented by a single output unit as implemented by the feature labels in the ISC-CI model.

We generate the episodes using the objects and features in the Leuven Concepts Database (De Deyne & Storms, 2008; Storms, 2001; Ruts et al., 2004). That database contains a matrix of binary judgments provided by human raters indicating, for each object-feature pairing, whether the object possess the feature (e.g., does a bear weigh more than 100lbs? Are kangaroos found in zoos?).

Let's explore the dataset


In [3]:
FEATURE_PATH = 'https://raw.githubusercontent.com/PrincetonUniversity/NEU-PSY-502/refs/heads/main/data/isc_ci/features.csv'

feature_df = pd.read_csv(FEATURE_PATH, index_col=0)

feature_df.head()

,is small,is a bird,is an animal,is big,can fly,is an insect,mammal,is a fish,lays eggs,is brown,...,is for all ages,you can play different notes whit it,can be used to put something in,can be bought in sports store,costs a lot of money,drives above the ground,driven by 1 person,used in water,used in the house,worn often
monkey,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
beaver,0,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
bison,0,0,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
dromedary,0,0,1,1,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
squirrel,1,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


Exercise 1a{exercise}

As stated above, a context is defined by the common feature shared by the objects in that context. For this dataset, which objects might occur in the "eats mice" context?

Solution{solution}

We can find the objects that might occur in the "eats mice" context by filtering the dataset for the feature "eats mice" and extracting the corresponding object names:

```python
eats_mice = feature_df[feature_df['eats mice'] == 1].index.tolist()
```

Exercise 1b{exercise}

As stated above, encountering different objects in the same episode provides evidence to the agent that they are in a specific context. For this dataset, which context might be defined by the object "owl" (and which by "falcon")?

Solution{solution}

The contexts are defined by the features of the objects:

```python
contexts_owl = feature_df.columns[feature_df.loc['owl'] == 1]
contexts_falcon = feature_df.columns[feature_df.loc['falcon'] == 1]

print('Contexts defined by "owl":', contexts_owl)
print('Contexts defined by "falcon":', contexts_falcon)
```



Exercise 1c{exercise}

Objects can appear in various contexts. So for two given objects, they might define multiple contexts. What possible contexts are defined by "owl" and "falcon" together?

Solution{solution}

The possible contexts are defined by the features that are shared by "owl" and "falcon":

```python
contexts_both = feature_df.columns[(feature_df.loc['owl'] == 1) & (feature_df.loc['falcon'] == 1)]
print(contexts_both)
```


Exercise 1d{exercise}

Now (using the theory from above), try to answer the following question: Given an agent experiences an episode with "owl" and "falcon" defining the context. Would the agent be more surprised by encountering a "cat" in this context or "penguin"?

(This is not a straight forward question, and you are not supposed to give a definite answer. Think about different (probabilistic/statistical) features of the world influencing the agent's prediction).

Tip: Here we make the (unreasonable) assumption that contexts are uniformly distributed. In other words that any episode (being on a bird-watching tour or being in a science lesson) has occurred equally often in the past.

Hint a{hint}

In the exercise above, we've seen that "owl" and "falcon" can elicit different contexts. But is there a way to "quantify" which of these contexts is more likely to be elicited?

Tip: This depends on how likely it is to encounter both an "owl" and a "falcon" in any of the given contexts.

Hint b{hint}

We calculate the probabilities from above:

```python
contexts_both = feature_df.columns[(feature_df.loc['owl'] == 1) & (feature_df.loc['falcon'] == 1)]

context_probabilities = {}

for c in contexts_both:
    objects = feature_df[feature_df[c] == 1].index.tolist()
    nr_objects = len(objects)
    # Simplification (objects are equally likely):
    # The probability for encountering any object is 1 / nr_objects
    # => the probability of 'drawing' two objects is 1 / nr_objects^2
    probability = 1 / nr_objects**2
    context_probabilities[c] = probability

print(context_probabilities)
print()
most_probabl_context = max(context_probabilities, key=context_probabilities.get)
print(most_probabl_context)
```

Hint c{hint}

The code above shows that the most probable context is "eats mice". But "penguins" don't eat mice while "cats" do (You can convince yourself by querying the database :). So if the agent encounters a "cat" it should be less surprised than if it encounters a "penguin".

However, this is just the most probable context. Although "penguin" doesn't appear in the most probable context of "owl" and "falcon", this might be offset by "penguin" appearing in more probable contexts than "cat".

Solution{solution}

First, we calculate the probabilities of encountering "owl" and "falcon" in any given context. We normalize these probabilties and use them as weights:
We add all the probabilities of contexts where "penguin" also appears vs "cat":

```python
contexts_both = feature_df.columns[(feature_df.loc['owl'] == 1) & (feature_df.loc['falcon'] == 1)]

context_probabilities = {}

for c in contexts_both:
    objects = feature_df[feature_df[c] == 1].index.tolist()
    nr_objects = len(objects)
    probability = 1 / nr_objects**2
    context_probabilities[c] = probability

# normalize the to get the probabilty of a certain context beeing evoked
sum_probs = sum(context_probabilities.values())
normalized = {c: p / sum_probs for c, p in context_probabilities.items()}

penguin_weight = 0
penguin_nr = 0
cat_weight = 0
cat_nr = 0


for c, v in normalized.items():
    objects = feature_df[feature_df[c] == 1].index.tolist()
    # the context evokes penguin
    if 'penguin' in objects:
        penguin_weight += v
        penguin_nr += 1
    # the context evokes cat
    if 'cat' in objects:
        cat_weight += v
        cat_nr += 1

print(f'penguins({penguin_nr}): {penguin_weight}')
print(f'cats({cat_nr}): {cat_weight}')
```

### Embedding

The ISC-CI model uses as input a word embedding (not a one-hot encoding). This embedding can be interpreted as "context independent" representation. Here, we load the embeddings:

In [35]:
EMBEDDING_PATH = 'https://raw.githubusercontent.com/PrincetonUniversity/NEU-PSY-502/refs/heads/main/data/isc_ci/embeddings.csv'

embeddings_df = pd.read_csv(EMBEDDING_PATH, index_col=0)

chicken_embedding = embeddings_df.loc['chicken']
chicken_embedding

0     0.119731
1     0.987248
2     0.982023
3     0.410961
4     0.451753
        ...   
59    0.022675
60    0.326714
61    0.014851
62    0.035483
63    0.982558
Name: chicken, Length: 64, dtype: float64

Exercise 2{exercise}

We can ask the same question as above: Can you think about a way of using the embeddings of "owl", "falcon", "cat" and "penguin" to quantify weather the agent would be more surprised by encountering a "cat" or a "penguin" in the context of "owl" and "falcon"?

Hint{hint}

With word embeddings, we can calculate similarities between words by using the distance.

Solution{solution}

We calculate the distance between the embeddings of "owl" and "falcon" to the embeddings of "cat" and "penguin":

```python
owl_emb = embeddings_df.loc['owl']
falcon_emb = embeddings_df.loc['falcon']
cat_emb = embeddings_df.loc['cat']
penguin_emb = embeddings_df.loc['penguin']

owl_falcon_emb = (owl_emb + falcon_emb) / 2

cat_distance = ((owl_falcon_emb - cat_emb)**2).sum()
penguin_distance = ((owl_falcon_emb - penguin_emb)**2).sum()

print('cat distance ', cat_distance)
print('penguin distance ', penguin_distance)
```

Note, the "context"-calculations and "embedding"-calculations lead to different predictions:

- context -> "cat" is less surprising in "owl", "falcon" context
- embedding  -> "penguin" is less surprising in "owl", "falcon" context

### Training Data